# Coin toss graph

In [15]:
import cv2
import numpy as np
import random
import time

In [25]:
def draw_graph(points, y_tick_labels, title_text, reference_y=None, window_name="Graph", width=800, height=600):
    """
    Draws and displays a real-time line graph in an OpenCV window.

    Args:
        points: List of (x, y) tuples, normalized to [0.0, 1.0].
        y_tick_labels: List of strings for Y-axis ticks (e.g., ["0.0", "0.5", "1.0"]).
        title_text: Multi-line string (use \\n) to show at the top.
        reference_y: Optional float in [0.0, 1.0]. Draws a horizontal reference line at this Y level.
        window_name: Name of the OpenCV window.
        width, height: Canvas size in pixels.
    """
    margin = 50
    graph_width = width - 2 * margin
    graph_height = height - 2 * margin

    # Create fresh white canvas
    canvas = np.ones((height, width, 3), dtype=np.uint8) * 255

    # Draw axes
    cv2.line(canvas, (margin, margin), (margin, height - margin), (0, 0, 0), 2)  # Y
    cv2.line(canvas, (margin, height - margin), (width - margin, height - margin), (0, 0, 0), 2)  # X

    # Y-axis ticks and labels
    num_ticks = len(y_tick_labels)
    for i, label in enumerate(y_tick_labels):
        y_norm = i / (num_ticks - 1) if num_ticks > 1 else 0.0
        y_pixel = int(height - margin - y_norm * graph_height)
        cv2.line(canvas, (margin - 5, y_pixel), (margin + 5, y_pixel), (200, 200, 200), 1)
        cv2.putText(canvas, label, (10, y_pixel + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 100, 100), 1)

    # Draw reference line (e.g., y = 0.5 for fair coin)
    if reference_y is not None and 0.0 <= reference_y <= 1.0:
        ref_y_pixel = int(height - margin - reference_y * graph_height)
        cv2.line(canvas, (margin, ref_y_pixel), (width - margin, ref_y_pixel), (180, 180, 180), 1, cv2.LINE_AA)

    # Draw data line
    if len(points) > 1:
        pixel_points = []
        for x_norm, y_norm in points:
            x = int(margin + x_norm * graph_width)
            y = int(height - margin - y_norm * graph_height)
            pixel_points.append((x, y))

        for i in range(len(pixel_points) - 1):
            cv2.line(canvas, pixel_points[i], pixel_points[i + 1], (255, 0, 0), 2, cv2.LINE_AA)

        cv2.circle(canvas, pixel_points[-1], 4, (0, 0, 255), -1)

    # Draw title
    y_offset = 30
    for i, line in enumerate(title_text.split('\n')):
        cv2.putText(canvas, line, (margin + 10, y_offset + i * 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 150), 2)

    # Show window
    cv2.imshow(window_name, canvas)


In [26]:
INTERVAL = 0.1
heads_count = 0
total_tosses = 0
ratios = []

try:
    while True:
        # Simulate one toss
        toss = random.randint(0, 1)
        heads_count += toss
        total_tosses += 1
        ratio = heads_count / total_tosses
        ratios.append(ratio)

        # Normalize data: x = index / (N-1), y = ratio
        if len(ratios) > 1:
            max_i = len(ratios) - 1
            points = [(i / max_i, r) for i, r in enumerate(ratios)]
        else:
            points = [(0.0, ratios[0])] if ratios else [(0.0, 0.0)]

        # Y-axis labels: 0.0 to 1.0
        y_labels = [f"{i/10:.1f}" for i in range(11)]

        # Title
        title = (
            f"Total tosses: {total_tosses}\n"
            f"Heads: {heads_count}\n"
            f"Ratio (H/T): {ratio:.4f}\n"
            "Press 'q' to quit"
        )

        # Draw graph
        draw_graph(
            points=points,
            y_tick_labels=y_labels,
            title_text=title,
            reference_y=0.5,
            window_name="Coin Toss Ratio"
        )

        # Exit check
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        time.sleep(INTERVAL)

        # Speed up every 100 tosses
        if total_tosses % 100 == 0:
            INTERVAL = max(INTERVAL / 2, 0.0001)  # Avoid zero sleep

except KeyboardInterrupt:
    pass
finally:
    cv2.destroyAllWindows()

# Dice roll graph

In [28]:
INTERVAL = 0.1
total_sum = 0
total_rolls = 0
averages = []

try:
    while True:
        # Simulate
        roll = random.randint(1, 6)
        total_sum += roll
        total_rolls += 1
        average = total_sum / total_rolls
        averages.append(average)

        # Normalize data for plotting: x in [0,1], y scaled to [0,1] for drawing
        if len(averages) > 1:
            max_i = len(averages) - 1
            points = [(i / max_i, (a - 1) / 5) for i, a in enumerate(averages)]
        else:
            avg = averages[0] if averages else 3.5
            points = [(0.0, (avg - 1) / 5)]

        # Y-axis labels: show actual values 1.0 to 6.0
        y_labels = [f"{val:.1f}" for val in np.linspace(1.0, 6.0, 11)]  # 1.0, 1.5, ..., 6.0

        # Title text
        title = (
            f"Total rolls: {total_rolls}\n"
            f"Sum: {total_sum}\n"
            f"Running average: {average:.4f}\n"
            "Press 'q' to quit"
        )

        # Draw graph
        draw_graph(
            points=points,
            y_tick_labels=y_labels,
            title_text=title,
            reference_y=0.5,
            window_name="Dice Roll Average"
        )

        # Exit check
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        time.sleep(INTERVAL)

        # Speed up every 100 rolls
        if total_rolls % 20 == 0:
            INTERVAL = max(INTERVAL / 2, 0.0001)

except KeyboardInterrupt:
    pass
finally:
    cv2.destroyAllWindows()

---------